# Realistic Example: Predicting House Prices

A complete walkthrough using everything from Day 9: `nn.Module`, `nn.Sequential`, dropout, training, evaluating.

### The problem

You have data about 500 houses (size, bedrooms, age, etc.) and their selling prices. **Can a neural network predict the price of a new house from its features?**

### What we'll do — step by step

```
1.  Generate realistic house data
2.  Explore the data (always do this first!)
3.  Split into train / validation / test
4.  Normalize features (critical step!)
5.  Build SIMPLE model (Day 1-3 style — just to remember why we left it)
6.  Build BETTER model with nn.Module (the Day 9 way)
7.  Train it
8.  Evaluate it
9.  Use it to predict prices for new houses
10. Inspect what each layer learned
```

This is a regression task (predicting a number, not a class). It's the classic "first real ML project."

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt

torch.manual_seed(42)
np.random.seed(42)

## Step 1: Generate Realistic House Data

Let's create 500 fake-but-realistic houses. Each house has 5 features:
- **size_sqft** — square footage (500-3500)
- **bedrooms** — number of bedrooms (1-6)
- **age_years** — how old the house is (0-50)
- **distance_to_city_km** — how far from the city center (0-30)
- **garage_spaces** — number of car spots (0-3)

The price is determined by a hidden formula (with noise) — your network has to figure it out.

In [ ]:
# Generate 500 houses

n = 500

size_sqft = np.random.uniform(500, 3500, n)
bedrooms = np.random.randint(1, 7, n)
age_years = np.random.uniform(0, 50, n)
distance_to_city_km = np.random.uniform(0, 30, n)
garage_spaces = np.random.randint(0, 4, n)

# Hidden price formula (the model has to learn this!):
#   price = $50 per sqft + $10k per bedroom - $2k per year of age
#         - $5k per km from city + $8k per garage space + base $50k + noise
price = (
    50 * size_sqft
    + 10_000 * bedrooms
    - 2_000 * age_years
    - 5_000 * distance_to_city_km
    + 8_000 * garage_spaces
    + 50_000
    + np.random.normal(0, 20_000, n)   # noise
)

# Stack features into a (n, 5) matrix
X = np.column_stack([size_sqft, bedrooms, age_years, distance_to_city_km, garage_spaces])
y = price

feature_names = ['size_sqft', 'bedrooms', 'age_years', 'distance_km', 'garages']

print(f"Shape of X: {X.shape}  (500 houses × 5 features)")
print(f"Shape of y: {y.shape}\n")

print("First 5 houses:")
print(f"{'sqft':>7} {'beds':>5} {'age':>4} {'dist':>5} {'gar':>4} | {'price':>10}")
print("-" * 55)
for i in range(5):
    print(f"{X[i,0]:>7.0f} {X[i,1]:>5.0f} {X[i,2]:>4.0f} {X[i,3]:>5.1f} {X[i,4]:>4.0f} | ${y[i]:>9,.0f}")

## Step 2: Explore the Data

ALWAYS look at your data before training. Skipping this step is the #1 cause of bugs in real projects.

In [ ]:
# Plot each feature vs price to see relationships

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

for i, name in enumerate(feature_names):
    ax = axes[i]
    ax.scatter(X[:, i], y / 1000, alpha=0.4, s=20)
    ax.set_xlabel(name)
    ax.set_ylabel('Price ($1000s)')
    ax.set_title(f'{name} vs price')
    ax.grid(True, alpha=0.3)

# Hide the 6th plot (we only have 5 features)
axes[5].set_visible(False)

plt.tight_layout()
plt.show()

# Print summary statistics
print("Feature ranges:")
print(f"{'Feature':>15} | {'min':>8} | {'max':>8} | {'mean':>8}")
print("-" * 55)
for i, name in enumerate(feature_names):
    print(f"{name:>15} | {X[:,i].min():>8.1f} | {X[:,i].max():>8.1f} | {X[:,i].mean():>8.1f}")
print(f"{'price':>15} | {y.min():>8,.0f} | {y.max():>8,.0f} | {y.mean():>8,.0f}")

### Notice the problem

Look at the feature ranges:
- `size_sqft` goes 500-3500 (huge numbers)
- `bedrooms` goes 1-6 (tiny numbers)
- `garages` goes 0-3 (also tiny)

If we feed this directly to a neural network, **size_sqft will dominate** because its values are 500x bigger than bedrooms. The model will struggle to learn the importance of bedrooms.

**Fix: normalization.** We scale each feature to mean=0, std=1.

---

## Step 3: Split Data — Train / Validation / Test

Standard 70 / 15 / 15 split:

In [ ]:
# Shuffle and split

shuffled = np.random.permutation(n)
X_shuf = X[shuffled]
y_shuf = y[shuffled]

# 70% train, 15% val, 15% test
train_end = int(0.7 * n)        # 350
val_end = int(0.85 * n)         # 425

X_train_raw = X_shuf[:train_end]
y_train_raw = y_shuf[:train_end]
X_val_raw = X_shuf[train_end:val_end]
y_val_raw = y_shuf[train_end:val_end]
X_test_raw = X_shuf[val_end:]
y_test_raw = y_shuf[val_end:]

print(f"Train: {len(X_train_raw)} houses")
print(f"Val:   {len(X_val_raw)} houses")
print(f"Test:  {len(X_test_raw)} houses")

## Step 4: Normalize Features

The standard recipe:
```
normalized = (value - mean) / std
```

After this, each feature has mean=0 and std=1.

**Critical:** Compute mean/std from TRAINING data only, then apply to all sets. Otherwise you're "cheating" by peeking at test data.

In [ ]:
# Compute mean and std from TRAIN data only
X_mean = X_train_raw.mean(axis=0)
X_std = X_train_raw.std(axis=0)

# Apply same normalization to all 3 sets
X_train = (X_train_raw - X_mean) / X_std
X_val = (X_val_raw - X_mean) / X_std
X_test = (X_test_raw - X_mean) / X_std

# Also normalize the target — makes training more stable
y_mean = y_train_raw.mean()
y_std = y_train_raw.std()
y_train = (y_train_raw - y_mean) / y_std
y_val = (y_val_raw - y_mean) / y_std
y_test = (y_test_raw - y_mean) / y_std

# Show the effect
print("BEFORE normalization (train data):")
print(f"  size_sqft → mean={X_train_raw[:,0].mean():.0f}, std={X_train_raw[:,0].std():.0f}")
print(f"  bedrooms  → mean={X_train_raw[:,1].mean():.1f}, std={X_train_raw[:,1].std():.1f}")
print(f"  → wildly different scales!")

print("\nAFTER normalization:")
print(f"  size_sqft → mean={X_train[:,0].mean():.2f}, std={X_train[:,0].std():.2f}")
print(f"  bedrooms  → mean={X_train[:,1].mean():.2f}, std={X_train[:,1].std():.2f}")
print(f"  → all features now on the same scale ✓")

# Convert to PyTorch tensors
X_train_t = torch.tensor(X_train, dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.float32).unsqueeze(1)  # (N,1)
X_val_t = torch.tensor(X_val, dtype=torch.float32)
y_val_t = torch.tensor(y_val, dtype=torch.float32).unsqueeze(1)
X_test_t = torch.tensor(X_test, dtype=torch.float32)
y_test_t = torch.tensor(y_test, dtype=torch.float32).unsqueeze(1)

## Step 5: Build the Model — `nn.Module` Style

This is where Day 9 pays off. Three layers with ReLU activations and dropout.

### Architecture:

```
Input (5 features)
   ↓
Linear (5 → 32) → ReLU → Dropout(0.2)
   ↓
Linear (32 → 32) → ReLU → Dropout(0.2)
   ↓
Linear (32 → 1) → output (predicted price)
```

In [ ]:
class HousePriceNet(nn.Module):
    """A small neural network for house price regression."""
    
    def __init__(self, input_dim=5, hidden_dim=32, dropout=0.2):
        super().__init__()
        # Layers are defined in __init__
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, hidden_dim)
        self.fc3 = nn.Linear(hidden_dim, 1)
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, x):
        # forward defines how data flows through layers
        x = F.relu(self.fc1(x))   # layer 1: linear → ReLU
        x = self.dropout(x)        # dropout (only active during training)
        x = F.relu(self.fc2(x))   # layer 2: linear → ReLU
        x = self.dropout(x)        # dropout
        x = self.fc3(x)            # output layer (no activation — raw price)
        return x

# Create the model
model = HousePriceNet(input_dim=5, hidden_dim=32, dropout=0.2)

print("Model architecture:")
print(model)
print(f"\nTotal parameters: {sum(p.numel() for p in model.parameters()):,}")
print()
print("Per-layer parameters:")
for name, param in model.named_parameters():
    print(f"  {name:>15}: shape {tuple(param.shape)} → {param.numel()} weights")

## Step 6: Train the Model

Standard training loop. Track both training and validation loss to spot overfitting.

In [ ]:
# Setup
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
loss_fn = nn.MSELoss()    # Mean Squared Error for regression

train_losses = []
val_losses = []

# Training loop
for epoch in range(300):
    
    # ---- TRAIN STEP ----
    model.train()                              # enables dropout
    pred = model(X_train_t)
    loss = loss_fn(pred, y_train_t)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    train_losses.append(loss.item())
    
    # ---- VALIDATION STEP ----
    model.eval()                               # disables dropout!
    with torch.no_grad():                      # no gradient tracking
        val_pred = model(X_val_t)
        val_loss = loss_fn(val_pred, y_val_t)
    val_losses.append(val_loss.item())
    
    if epoch % 50 == 0:
        print(f"Epoch {epoch:3d}: train_loss={loss.item():.4f}, val_loss={val_loss.item():.4f}")

print(f"\nFinal: train_loss={train_losses[-1]:.4f}, val_loss={val_losses[-1]:.4f}")

# Plot
plt.figure(figsize=(10, 4))
plt.plot(train_losses, 'b-', label='Train', linewidth=2)
plt.plot(val_losses, 'r-', label='Val', linewidth=2)
plt.xlabel('Epoch')
plt.ylabel('Loss (MSE on normalized prices)')
plt.title('Training Progress')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## Step 7: Evaluate on the Test Set

Test set = the data the model has NEVER seen. This is the real measure of how good the model is.

We'll compute:
- **MSE on normalized prices** (the loss the model trained on)
- **Mean error in DOLLARS** (what humans actually care about)
- **R²** — how well the predictions match (1.0 = perfect, 0 = no better than mean)

In [ ]:
model.eval()
with torch.no_grad():
    test_pred_normalized = model(X_test_t).squeeze().numpy()

# Convert back to actual dollar prices (un-normalize)
test_pred_dollars = test_pred_normalized * y_std + y_mean
test_actual_dollars = y_test_raw

# Metrics
errors_dollars = test_pred_dollars - test_actual_dollars
mae_dollars = np.abs(errors_dollars).mean()              # Mean Absolute Error
rmse_dollars = np.sqrt((errors_dollars ** 2).mean())     # Root Mean Squared Error

# R-squared: 1 = perfect, 0 = no better than predicting the mean
ss_res = np.sum(errors_dollars ** 2)
ss_tot = np.sum((test_actual_dollars - test_actual_dollars.mean()) ** 2)
r2 = 1 - ss_res / ss_tot

print(f"TEST SET RESULTS ({len(test_actual_dollars)} houses):")
print(f"  Mean Absolute Error: ${mae_dollars:>10,.0f}")
print(f"  RMSE:                ${rmse_dollars:>10,.0f}")
print(f"  R² score:            {r2:.4f}")
print()
if r2 > 0.95:
    print("  Excellent fit!")
elif r2 > 0.8:
    print("  Good fit!")
elif r2 > 0.5:
    print("  OK fit (could be better)")
else:
    print("  Poor fit (something might be wrong)")

# Plot predictions vs actuals
plt.figure(figsize=(8, 8))
plt.scatter(test_actual_dollars / 1000, test_pred_dollars / 1000, alpha=0.6, s=40, edgecolor='black')

# Perfect prediction line
lo = min(test_actual_dollars.min(), test_pred_dollars.min()) / 1000
hi = max(test_actual_dollars.max(), test_pred_dollars.max()) / 1000
plt.plot([lo, hi], [lo, hi], 'r--', linewidth=2, label='Perfect prediction')

plt.xlabel('Actual price ($1000s)')
plt.ylabel('Predicted price ($1000s)')
plt.title(f'Test set: predicted vs actual (R² = {r2:.3f})')
plt.legend()
plt.grid(True, alpha=0.3)
plt.axis('equal')
plt.tight_layout()
plt.show()

## Step 8: Use the Model to Predict New House Prices

Time for the fun part — let's predict prices for houses we just made up.

In [ ]:
def predict_house_price(size_sqft, bedrooms, age, distance_km, garages):
    """Predict the price of a new house."""
    # Build feature vector
    features = np.array([[size_sqft, bedrooms, age, distance_km, garages]])
    # Normalize using TRAIN stats
    features_norm = (features - X_mean) / X_std
    # Predict
    model.eval()
    with torch.no_grad():
        x = torch.tensor(features_norm, dtype=torch.float32)
        pred_norm = model(x).item()
    # Un-normalize
    pred_dollars = pred_norm * y_std + y_mean
    return pred_dollars

# Test on imaginary houses
houses_to_predict = [
    # (size, beds, age, dist, garages, description)
    (1500, 3, 5,  10, 1, "Average suburban home"),
    (3000, 5, 2,   5, 2, "Big new house close to city"),
    (800,  1, 30, 25, 0, "Tiny old house far from city"),
    (2500, 4, 10, 8,  2, "Family home, decent location"),
    (1200, 2, 15, 3,  1, "Small downtown apartment-style"),
]

print(f"{'Description':>40} | {'Predicted price':>18}")
print("-" * 65)
for size, beds, age, dist, gar, desc in houses_to_predict:
    price = predict_house_price(size, beds, age, dist, gar)
    print(f"{desc:>40} | ${price:>15,.0f}")

## Step 9: How Much Does Each Feature Matter?

Let's see what the model actually learned by changing one feature at a time.

In [ ]:
# For each feature, vary its value and see how price changes
# (keep all other features at their average)

baseline_features = X_train_raw.mean(axis=0)
baseline_price = predict_house_price(*baseline_features)

print(f"Baseline (average house): ${baseline_price:,.0f}")
print(f"  Avg size: {baseline_features[0]:.0f} sqft")
print(f"  Avg beds: {baseline_features[1]:.1f}")
print(f"  Avg age:  {baseline_features[2]:.1f} yrs")
print(f"  Avg dist: {baseline_features[3]:.1f} km")
print(f"  Avg gar:  {baseline_features[4]:.1f}")

# Show how each feature affects price
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

# What ranges to test for each feature
ranges = [
    np.linspace(500, 3500, 50),     # size
    np.arange(1, 7),                 # bedrooms
    np.linspace(0, 50, 50),          # age
    np.linspace(0, 30, 50),          # distance
    np.arange(0, 4),                 # garages
]

for i, (feat_range, name) in enumerate(zip(ranges, feature_names)):
    prices = []
    for value in feat_range:
        feats = baseline_features.copy()
        feats[i] = value
        prices.append(predict_house_price(*feats))
    
    axes[i].plot(feat_range, np.array(prices) / 1000, 'b-o', markersize=4)
    axes[i].set_xlabel(name)
    axes[i].set_ylabel('Predicted price ($1000s)')
    axes[i].set_title(f'How {name} affects price')
    axes[i].grid(True, alpha=0.3)

axes[5].set_visible(False)
plt.tight_layout()
plt.show()

print("\nThe model learned the right relationships:")
print("  size_sqft   ↑ → price ↑")
print("  bedrooms    ↑ → price ↑")
print("  age_years   ↑ → price ↓")
print("  distance    ↑ → price ↓")
print("  garages     ↑ → price ↑")

## Step 10: Save and Reload the Model

Now that we have a working model, let's save it so we don't have to retrain. This is what every real ML project does after training.

In [ ]:
# Save everything we need to use this model later

import os
os.makedirs('/tmp/house_model', exist_ok=True)

# Save the model weights
torch.save(model.state_dict(), '/tmp/house_model/weights.pth')

# Save the normalization stats (we need these to use the model later!)
np.savez('/tmp/house_model/normalization.npz',
         X_mean=X_mean, X_std=X_std, y_mean=y_mean, y_std=y_std)

print(f"Saved to /tmp/house_model/")
print(f"  weights.pth         — model parameters")
print(f"  normalization.npz   — feature scaling stats")

# Now reload everything in a "fresh" model
fresh_model = HousePriceNet(input_dim=5, hidden_dim=32, dropout=0.2)
fresh_model.load_state_dict(torch.load('/tmp/house_model/weights.pth'))
fresh_model.eval()

stats = np.load('/tmp/house_model/normalization.npz')
loaded_X_mean = stats['X_mean']
loaded_X_std = stats['X_std']
loaded_y_mean = stats['y_mean']
loaded_y_std = stats['y_std']

# Verify: predict the same as before
test_house = np.array([[1500, 3, 5, 10, 1]])
test_house_norm = (test_house - loaded_X_mean) / loaded_X_std
with torch.no_grad():
    pred_norm = fresh_model(torch.tensor(test_house_norm, dtype=torch.float32)).item()
pred = pred_norm * loaded_y_std + loaded_y_mean

print(f"\nFresh-loaded model predicts: ${pred:,.0f} for a 1500sqft, 3-bed, 5-yr-old, 10km, 1-garage house")
print(f"This should match the prediction from earlier!")

## Recap: What Just Happened

You built a complete, working ML system from scratch. Walking through the steps:

```
1.  GENERATE DATA          → 500 houses with realistic features
2.  EXPLORE                → looked at feature distributions
3.  SPLIT                  → 70/15/15 train/val/test
4.  NORMALIZE              → mean=0, std=1 (KEY for stable training)
5.  BUILD MODEL            → nn.Module with Linear + ReLU + Dropout
6.  TRAIN                  → tracked train + val loss
7.  EVALUATE               → R² score, mean error in dollars
8.  PREDICT                → for new houses
9.  INTERPRET              → see what each feature does
10. SAVE/LOAD              → ready for production use
```

### Key things you used from Day 9:

- ✓ `nn.Module` base class
- ✓ `super().__init__()`
- ✓ `nn.Linear` for fully-connected layers
- ✓ `F.relu` for activation
- ✓ `nn.Dropout` for regularization
- ✓ `model.train()` / `model.eval()` for proper dropout behavior
- ✓ `model.state_dict()` for saving
- ✓ `torch.no_grad()` for evaluation/prediction

### What's missing (to graduate to real-world projects)

- **Real data** — actual house data from Kaggle / sklearn
- **More features** — neighborhood, schools, recent renovations
- **Better metrics** — log-error, MAPE
- **Hyperparameter tuning** — try different hidden sizes, learning rates
- **Bigger network** — for more complex relationships

But the **structure** of every regression project is what you just did. Plug in different data, tweak the architecture, and you have a working ML solution.

### Try yourself

- Change `hidden_dim` from 32 to 8. Does R² go down?
- Change `dropout` to 0.5. Does train loss go up but val loss stay similar?
- Remove normalization (use raw `X_train_raw`). Does training still work?
- Add a sixth feature like "has_pool" (binary 0/1) and see if the model uses it.